## Notebook 8: Dynamic CBA

<blockquote style="border-left:4px solid #ccc; padding-left:1em;">
Here we take the city-specific outputs from previous notebooks (hazard, exposure, GVI uplift, AC coverage, CLIMADA runs) and build time profiles of adaptation costs for trees and AC (CAPEX + O&M), time profiles of benefits (avoided heat-related deaths), aggregate everything over a 25-year horizon with a 3% discount rate and derive equivalent annual costs (EACs) and cost-per-avoided-death metrics
</blockquote>

In [15]:
import os
os.environ["CITY"] = "rome"   # pick the city here

In [16]:
# Generic bootstrap 
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup import bootstrap
from cityheat.paths import make_P, ensure_out

# Choose city here
SLUG = globals().get("SLUG", os.environ.get("CITY", "rome")).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")

→ City: rome  |  BASE=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome  OUT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome  INT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim


In [17]:
# city config + file paths from YAML 
cfg = C.get("cfg", {})                      # full YAML for the selected city
SLUG = cfg.get("slug", SLUG).lower()        
CITY = cfg.get("city_name", CITY)

paths    = cfg.get("files", {})             # {gvi_csv, lcz_candidates, cooling_coeffs_csv, ...}
osm_cfg  = cfg.get("osm", {})               # OSM settings used later in NB5
trees_cfg = cfg.get("trees", {})            # TARGET/CAP for NB5
urbclim   = cfg.get("urbclim", {})          # UrbClim folder/settings for NB5

lcz_candidates = [P(p) for p in paths.get("lcz_candidates", [])]
gvi_path = P(paths.get("gvi_csv", ""))

# FUA geopackage written in NB2 
fua_gpkg = Path(paths.get("fua_gpkg", f"{OUT}/{SLUG}_fua.gpkg"))

cool_csv = P(paths.get("cooling_coeffs_csv", ""))

# checks
print("SLUG/CITY:", SLUG, CITY)
print("GVI CSV:  ", gvi_path)
print("LCZ cand: ", [str(p) for p in lcz_candidates])
print("FUA GPKG: ", fua_gpkg)
print("Cooling CSV:", cool_csv)

SLUG/CITY: rome Rome
GVI CSV:   /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/gviRome/gvi_Rome.csv
LCZ cand:  ['/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_filter_v3.tif', '/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_v3.tif']
FUA GPKG:  /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/rome_fua.gpkg
Cooling CSV: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/CoolingEff/outer_2_wbgt_max.csv


**Loading from before**

We now load all the inputs that previous notebooks produced:
- trees_tbl: municipio-level tree policy and resulting ΔGVI
- veg_diag: diagnostic JSON with citywide pop-weighted ΔGVI (1–100 index)
- ac_coverage_maps_{SLUG}.npz: baseline vs policy AC coverage on the grid
- pop_on_ref_{SLUG}.npz: population on the common reference grid
- muni_cov_{SLUG}.csv: municipio-level AC coverage (baseline vs policy)
- {SLUG}_muni_ac_consumption_summary.csv: kWh per AC user by municipio, year
- ac_eff_buckets_{SLUG}.json: relative AC efficacy by age class (for benefits)

In [18]:

# loading everything needed for the CBA
from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT = Path(OUT)
INT = Path(INT)
TAB_DIR = OUT / "tables"

# Vegetation policy / ΔGVI
trees_tbl = pd.read_csv(TAB_DIR / f"{SLUG}_trees_tbl.csv")

# diagnostics JSON with citywide ΔGVI (pop-weighted, points on 1–100 scale)
veg_diag_path = OUT / f"{SLUG}_veg_diagnostics.json"
veg_diag = json.loads(veg_diag_path.read_text())
citywide_dGVI_points_popw = veg_diag["citywide_dGVI_points_popw"]
print("Citywide pop-weighted ΔGVI (points, 1–100 scale):", citywide_dGVI_points_popw)

# AC coverage / municipal pop / energy use
# coverage maps
ac_cov_npz = np.load(INT / f"ac_coverage_maps_{SLUG}.npz")
coverage_base = ac_cov_npz["coverage_base"]
coverage_policy = ac_cov_npz["coverage_policy"]
CITY_MASK = ac_cov_npz["CITY_MASK"].astype(bool)
HGT = int(ac_cov_npz["HGT"])
WDT = int(ac_cov_npz["WDT"])

# population on ref grid
pop_npz = np.load(INT / f"pop_on_ref_{SLUG}.npz")
pop_on_ref = pop_npz["pop"]

# municipio coverage table (pop_muni, ac_base_muni, ac_policy_muni)
muni_cov = pd.read_csv(OUT / f"muni_cov_{SLUG}.csv")

# AC consumption per municipality (kWh per AC user)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")

# AC efficacy by age (for benefits, not directly cost-side)
eff_buckets_path = INT / f"ac_eff_buckets_{SLUG}.json"
EFF_BUCKETS = json.loads(eff_buckets_path.read_text())
EFF_BUCKETS

Citywide pop-weighted ΔGVI (points, 1–100 scale): 2.3197


{'<15': 0.2, '15-64': 0.3, '65+': 0.4}

**Discount helpers**

We assume:
- Time horizon T = 25 years
- Constant real discount rate r = 3%
- All flows occur at the end of each year (years 1..T).

- Helper functions:
    - pv_level_flow: present value (PV) of a constant annual flow over T years.
    - pv_replacements: PV of buying an item at t=0 and then replacing it every 'life'
      years within the horizon.
    - annuity_factor: present value of "1 euro per year" over T years. We later use this
      to convert any PV into a constant equivalent annual cost (EAC).
    - pv_capex_with_ramp: PV of AC capex when new users are added year by year(ramp-up),
      with replacements every `life` years.

- Intuition:
    - First we build the actual time path of costs (trees and AC).
    - Then we discount and sum to get a PV.
    - Finally, we divide the PV by the annuity factor to get a flat annual amount that is
      financially equivalent to the time-varying cashflow.

In [19]:
import numpy as np

R = 0.03 # discount rate
T = 25 # time horizon

# Convention: all flows occur at END of each year => years 1..T

# present value (PV) of a constant annual flow over T years.
def pv_level_flow(annual, r=R, T=T):
    """PV of a constant annual amount paid in years 1..T."""
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(annual * (1 + r) ** (-yrs)))

# PV of buying an item at t=0 and then replacing it every 'life' years within the horizon.
def pv_replacements(n_items, capex_per_item, r=R, T=T, life=20):
    """ PV of buying 'n_items' at t=0 and replacing every 'life' years within horizon T. """
    pv = 0.0
    t = 0
    while t <= T:
        pv += n_items * capex_per_item / ((1+r)**t if t > 0 else 1.0)
        t += life
    return float(pv)

# annuity_factor: present value of "1 euro per year" over T years. We later use this
# to convert any PV into a constant equivalent annual cost (EAC).
def annuity_factor(r=R, T=T):
    return (1 - (1 + r) ** (-T)) / r

AF = annuity_factor(R, T)
AF

17.413147691278027

In [20]:
def pv_capex_with_ramp(new_users_t, capex_per_user, life, r=R):
    """ PV of AC capex when policy coverage ramps up over time.

    new_users_t: 1D array length T, number of new policy users in each year (vs baseline).
    capex_per_user: installation cost per AC user.
    life: years between replacements.
    r: discount rate.

    For each cohort of new users in year t, we:
    - pay capex once at installation (year t+1 in our convention)
    - then pay the same capex again every life years (replacement)
    - discount each of these payments back to year 0 and sum them up.
    """
    T = len(new_users_t)
    pv = 0.0
    for t in range(T):  # t = 0..T-1 corresponds to years 1..T
        cohort = float(new_users_t[t])
        if cohort <= 0:
            continue
        pay_year = t  # installation in year t+1, then every 'life' years
        while pay_year < T:
            pv += cohort * capex_per_user / ((1 + r) ** (pay_year + 1))
            pay_year += life
    return float(pv)

**Trees: parametrisation of costs**

Some explanation on following code:
- Total index : how many GVI index points we need in total (once and for all) to reach policy target
- Investment rule: each new index point costs 10M once (investment), not every year. Spreading creation of those index points linearly over 25y : each year we create the same slice of change in GVI and pay 10M*that slice in that year. That's the change in GVI = sum(change in GVI/25) and 10M * change in GVI/25 per year.
- CAPEX PV: npv_capex_linear: takes ramp of yearly investments (same amount each year, over 25 years) and discounts them as flows in years 1...25. PV_trees_capex: NPV of the linear capex schedule
- IR vs EAC: TREES_CAPEX_T0: total undiscounted investment requirement
- Report EAC_capex_annuity=PV_trees_capex/AF : equivalent annual cost of our explicit linear ramp and EAC_capex_paper=TREES_CAPEX_T0/(1+R)^T*T) : as if all investments happens by T and we just annualise that discounted IR.
- O&M: npv_om_cohorts: treats O&M as yearly flows in years 1...25 each year we add a new cohort, and the number of active cohorts in year t is min(t, lifetime). We pay O&M per index point per year for all active cohorts and discount those flows. Matches description of O&M series that starts when trees are planted, accumulates cohorts, and is discounted. 

**Timing conventions for tree costs.**  
We consider a 25-year horizon and treat all cash flows as occurring at the end of each year (years 1–25). The total increase in the GVI index (ΔGVI, in 1–100 index points) implied by the tree policy is first converted into a **total investment requirement** using the rule that raising GVI by 1 index point costs 10 million euro once and for all. We then assume this investment is implemented along a **linear ramp**: the same fraction of ΔGVI is created in each year, and the corresponding investment is spread evenly over the 25 years. The functions `npv_capex_linear` and `npv_om_cohorts` compute the discounted present value of, respectively, this phased investment schedule and the recurrent O&M costs associated with overlapping planting cohorts. From these present values we derive equivalent annual costs (EACs) by dividing by the standard annuity factor. For comparison with the article (working paper), we also report a “paper-style” EAC where the total undiscounted investment requirement is pushed to the end of the horizon, discounted once, and then divided by the number of years.

**Trees: dynamic CAPEX + O&M costs for a given GVI uplift**

- Policy:
    - We simulate a tree-planting policy that increases the Green View Index (GVI)
    in low-GVI municipi up to a target level.
    - DELTA_INDEX is the total citywide increase in GVI, measured in 1–100 index points,
      implied by this policy (sum over municipi of dGVI * 100).
- Cost rule:
    - We use an empirical rule: increasing GVI by 1 index point costs €10 million
      once and for all (investment) (working paper)
    - TREES_CAPEX_T0 = 10 M€ * DELTA_INDEX is the undiscounted total investment
      requirement if we imagined doing all planting at once.

- Time pattern (ramp):
    - In reality we don't plant everything in one year. Instead we assume a
      linear rollout over T = 25 years:
      each year we create the same slice of ΔGVI (DELTA_INDEX / T)
      each slice pays CAPEX once at planting in that year
    - npv_capex_linear():
      builds this stream of annual investments (same € amount each year)
      discounts each year's CAPEX to year 0
      sums to a present value PV_trees_capex

- O&M (operation and maintenance):
    - Tree costs do not end at planting: each "cohort" of trees needs O&M every year.
    - We use REGREEN per-tree numbers (CAPEX = 210 €, O&M = 27 €/yr) to derive an
      O&M cost per GVI index point per year (OM_PER_INDEX_PT_YR).
    - npv_om_cohorts():
      assumes we add the same GVI increment each year (cohorts)
      each cohort pays OM_PER_INDEX_PT_YR * increment every year after planting
      up to LIFETIME_YEARS (25 here)
      at year t, there are min(t, LIFETIME_YEARS) active cohorts
      discounts the resulting O&M stream to year 0 to get PV_trees_om

- Aggregation and annualisation:
    - PV_trees_total = PV_trees_capex + PV_trees_om is the total present value of the
      tree programme (investment + O&M).
    - We convert each PV into an equivalent annual cost by dividing by the annuity factor
      AF:
      EAC_capex_annuity = PV_trees_capex / AF
      EAC_om_annuity    = PV_trees_om    / AF
      EAC_total_annuity = PV_trees_total / AF

- Paper-style EAC (for comparability only):
    - EAC_capex_paper uses the shortcut from the original paper:
  take TREES_CAPEX_T0 (total undiscounted investment requirement),
  pretend it is paid all at year T,
  discount once by (1 + r)^T,
  ivide by T.
    - This does NOT reflect our explicit linear rollout => gives an "average discounted annual investment" from a single IR number.
    - The economically consistent metric for our dynamic ramp is the annuity-based
    - EAC derived from PV_trees_capex and PV_trees_om.

In [29]:
# Parameters from rule and regreen study
CAPEX_PER_INDEX_PT = 10_000_000.0 # eur per 1 index point (1–100 scale)

# converting per-tree CAPEX and per-tree O&M into an O&M cost per GVI point.
CAPEX_PER_TREE = 210.0 # eur per tree (REGREEN median)
OM_PER_TREE_YR = 27.0 # eur per tree per year
LIFETIME_YEARS = 25 # tree benefit/O&M lifetime

# O&M per index point per year implied by tree-level numbers
OM_PER_INDEX_PT_YR = (OM_PER_TREE_YR / CAPEX_PER_TREE) * CAPEX_PER_INDEX_PT
print("O&M per index point per year (EUR):", round(OM_PER_INDEX_PT_YR, 0))

# Total change in GVI in index point (1–100 SCALE) from trees_tbl
# trees_tbl['dGVI'] is in 0–1 (fraction of max index); sum over Municipi, then ×100 => points
DELTA_INDEX = float(trees_tbl["dGVI"].clip(lower=0).sum()) * 100.0 # sum of municipio dGVI (0–1) => index points (0–100)
print("Total ΔGVI index points (1–100 scale):", round(DELTA_INDEX, 2))

# For comparison: citywide pop-weighted ΔGVI (diagnostics)
print("Pop-weighted ΔGVI points (diagnostic):", citywide_dGVI_points_popw)

# CAPEX: linear ramp over T years
# linear rollout over 25 years
# each year we add (ΔGVI / 25) points
# pay CAPEX once for that increment in that year
# investment schedule(t) = 10M€ * (ΔGVI/25) every year, discounted year by year.
def npv_capex_linear(delta_index_total, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT):
    """ PV of a linear ramp: we add delta_index_total/years index points each year over 'years',
    and pay capex_per_index per point. All flows are assumed at the end of years 1..years.
    """
    inc = delta_index_total / years  # index points added per year
    pv = 0.0
    for t in range(1, years + 1):  # t = 1..years
        capex_t = capex_per_index * inc
        pv += capex_t / ((1 + r) ** t)
    return float(pv)

# Paper-style "investment requirement" if all done at once (undiscounted)
TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX

PV_trees_capex = npv_capex_linear(DELTA_INDEX, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT)
print(f"Trees — Total CAPEX requirement (undiscounted): €{TREES_CAPEX_T0:,.0f}")
print(f"Trees — NPV CAPEX (linear ramp): €{PV_trees_capex:,.0f}")

# O&M: overlapping cohorts with constant per-index-point O&M
# each year we plant a new cohort (same GVI increment each year under linear rollout)
# each cohort generates an annual O&M cost in all subsequent years while active
# total O&M in year t is the sum across all cohorts still within their O&M lifetime
def npv_om_cohorts(delta_index_total, years=T, r=R,
                   om_per_index_per_year=OM_PER_INDEX_PT_YR, lifetime=LIFETIME_YEARS):
    """
    O&M with overlapping cohorts, but O&M starts the year AFTER planting.

    Each year we add 'inc' index points. A cohort planted in year k pays O&M in years k+1, k+2, ...
    All flows are assumed at the end of years 1..years.
    """
    inc = delta_index_total / years
    pv = 0.0
    for t in range(1, years + 1):       # t = 1..years
        active = min(t - 1, lifetime)    # exclude planting-year cohort
        om_t = active * om_per_index_per_year * inc
        pv += om_t / ((1 + r) ** t)
    return float(pv)

PV_trees_om = npv_om_cohorts(DELTA_INDEX, years=T, r=R, om_per_index_per_year=OM_PER_INDEX_PT_YR, lifetime=LIFETIME_YEARS)
print(f"Trees — NPV O&M (cohorts): €{PV_trees_om:,.0f}")

# Total PV of tree programme (consistent, because both CAPEX + O&M follow explicit timing)
PV_trees_total = PV_trees_capex + PV_trees_om
print(f"Trees — NPV total (CAPEX + O&M): €{PV_trees_total:,.0f}")

# 2 approaches of annualisation

# Annuity-based EAC (economically consistent with explicit cashflows)
# This says: "what constant yearly payment over 25y has the same PV as the actual schedule?"
AF = annuity_factor(R, T)
EAC_capex_annuity = PV_trees_capex / AF
EAC_om_annuity = PV_trees_om / AF
EAC_total_annuity = PV_trees_total / AF

# Article-like “investment requirement divided by (1+r)^T * T”
# which is equivalent to pretending the full IR happens at year T,
# discounting once, then spreading evenly across years.
EAC_capex_paper = TREES_CAPEX_T0 / ((1 + R)**T * T)

print(f"Trees — EAC CAPEX (annuity): €{EAC_capex_annuity:,.0f}/yr")
print(f"Trees — EAC O&M (annuity): €{EAC_om_annuity:,.0f}/yr")
print(f"Trees — EAC total (annuity): €{EAC_total_annuity:,.0f}/yr")
print(f"Trees — EAC CAPEX (paper-style): €{EAC_capex_paper:,.0f}/yr")

O&M per index point per year (EUR): 1285714.0
Total ΔGVI index points (1–100 scale): 30.23
Pop-weighted ΔGVI points (diagnostic): 2.3197
Trees — Total CAPEX requirement (undiscounted): €302,333,765
Trees — NPV CAPEX (linear ramp): €210,583,300
Trees — NPV O&M (cohorts): €283,658,615
Trees — NPV total (CAPEX + O&M): €494,241,914
Trees — EAC CAPEX (annuity): €12,093,351/yr
Trees — EAC O&M (annuity): €16,289,910/yr
Trees — EAC total (annuity): €28,383,261/yr
Trees — EAC CAPEX (paper-style): €5,775,852/yr


- DELTA_INDEX: citywide change in GVI on 1-100 scale
- inc = DELTA_INDEX/years is change in GVI / 25 each year => linear path
- capex_t = 10Meur * inc is the 10Meur*change in GVI/25 rule
- Discounting stream year by year to get PV_trees_capex
- TREES_CAPEX_T0: paper way of doing it: what if we did everything upfront

- each year we plant inc index points (new cohort)
- each cohort costs om_per_index_per_year * inc every year
- after t years, there are t+1 overlapping cohorts (until we cap at lifetime)
- om_t is exactly sum over all active cohorts
- then we discount om_t year by year
- here, we choose that all cohorts have same per-index-point O&M each year for lifetime years

BIG QUESTION HERE BECAUSE I NEVER UNDERSTAND:
About the annualisation:
- We have two different annualisation ideas in the code
- 1. Economically consistent one with our ramp (PV_trees_capex, AF, EAC_capex_annuity)
     Interpretation: npv_capex_linear gives the present value of the phased CAPEX stream
     Dividing by the annuity factor (that only depends on T and r), converts that pV into
     a constant equivalent annual cost over 25 years.
     Basically, given a specific investment path (here: linear ramp), we discount it,
     then convert to PV into a flat annual amount.
  2. EAC_capex_paper: this is not the same thing as annuity based on ramp. It's like the
     paper of Giacomo: take undiscounted total investment requirement IR, push it to year
     T, discount it once, divide by T. It's to get the "average discounted annual cost"
     but assumes everything at the end and does not reflect our explicit tamp path. The
     correct one should be the 1. but need to ask Giacomo more about this. 

More explanation about this...
- the annuity EAC: takes the actual NPV of the ramped CAPEX (and O&M) streams, divide by the standard annuity factor. "Constant yearly cost, over 255 years, that is financially equivalent to this time varying cashflow"
- Paper : take the total undiscounted investment requirement (as if all invested once), shift it to year T then divide by T. It doesn't reflect our ramp. It's a shortcut to get an average discounted annual cost from a single IR number. We use it for comparability.

From what I understand, what I do is what we discussed: we have a total increase in GVI needed (delta_index), we assume a linear path over 25 years and each year we add change in GVI/25 index points. We use the rule: 1 index points: 10M once, not every year. So each year we invest 10M*(deltaGVI/25). We discount that year by year to get an NPV of capex: npv_capex_linear. For O&M, cohort logic: each year, new cohort planted, each cohort pays the same O&M per index point each year, in year t we have mint(, lifetime) active cohorts, we discount the whole panel of O&M flows: npv_om_cohorts. We do the standard annuity step : taking NPV of CAPEX (or total CAPEX + O&M) and dividing by annuity factor sum from t=1 to T of (1+r)^t to get a constant yearly cost that is financially equivalent to the detailed schedule. 

- npv_capex_linear: discounted sum of future cashflows with a linear ramp
- npv_om_cohorts: O&M cohorts idea, discounted year by year
- EAC by dividing by annuity factor. Depends only on r and T because it’s the present value of paying “1€ every year” over T years. It doesn’t have to know about how cohorts build up; that’s already into the NPV. 

What is happening in the paper? IRc is a total undiscounted investment requirement (already aggregated over the 25 y horizon). Then, you push that entire amount to year T, discount it once by (1+r)^T and divide by T. It's: taking the total investment we would need in today's euros, pretending it's all paid in year T, discounting that once, then just spreading the discounted lump evenly over T years. 

In the paper, the formula gave an “average discounted yearly investment” directly from a total IR, without specifying a time pattern.

Here, we now do specify the time pattern explicitly (linear ramp of ΔGVI and overlapping O&M cohorts). So I compute NPV as the discounted sum of those yearly cashflows, and then convert that NPV into a constant equivalent annual cost using the standard annuity factor for r = 3% and T=25.

This annuity-based EAC is the one that’s fully consistent with the dynamic ramp and with the way we treat benefits (yearly avoided deaths). I still report the paper-style EAC (IR/(1+r)^T/T) alongside, to keep direct comparability with your article.

**Sensitivity paved streets**

TO DO IF IT'S RIGHT, WITH 5379.0 instead of 210.

**Cost AC**

**Air conditioning (AC): dynamic coverage and dynamic electricity use** 

- Policy:
    - We expand AC coverage in under-served, poorer CAPs up to a target share.
    - The result is a time series of incremental AC coverage by municipio and year
      (policy vs baseline), coming from previous notebooks.
      
- Two key dynamics:
- 1) Coverage ramp:
    - muni_cov_all gives AC coverage for 2030, 2040, 2050 by municipio under baseline and
     policy.
    - For each municipio, we compute the additional AC share (policy - baseline) and
      interpolate it linearly year-by-year over 2030–2054.
    - Multiplying by pop_muni gives the number of additional AC users per year.
        - added_users_t = total incremental AC users in each year (vs baseline).
        - new_users_t   = number of *new* users added each year (increments of
          added_users_t) => drives CAPEX cohorts.
- 2) Electricity use per user:
     - NB7 gives kWh per AC user at city and municipio level for 2030, 2040, 2050.
     - We linearly interpolate kWh per user by municipio for each year of the
       25-year horizon (2030–2054).
     - For each municipio and year we then compute: kWh = pop_muni * extra AC share * kWh
       per user and aggregate to the city.
     
Cost components:
- CAPEX:
    - AC_CAPEX_PER_USER is the unit cost per new AC user.
    - pv_capex_with_ramp(new_users_t, ...) computes the present value of capex for each
      cohort, including replacements every AC_LIFETIME_YEARS.
- Maintenance:
    * Each incremental AC user pays a fixed fraction of CAPEX (5%) per year.
    * maint_eur_t = added_users_t * maint_per_user_yr gives yearly maintenance.
    * We discount this stream to get PV_ac_maint.
- Electricity:
    * elec_eur_t are yearly electricity costs (kWh * tariff) with both dynamic coverage
      and dynamic kWh/user.
    * We discount them to get PV_ac_elec.
      
- Aggregation and annualisation:
    - PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec.
    - EAC_ac_total = PV_ac_total / AF gives the equivalent constant annual cost over
      25 years of the full AC expansion (including CAPEX, maintenance and electricity).

- Interpretation:
- This AC module is fully dynamic: both the number of users and their per-user electricity consumption change over time, and CAPEX follows cohorts + replacements.
- This keeps AC costs consistent with the way we treat benefits (annual profiles over 25 years, discounted at 3%).

In [30]:
# Dynamic AC horizon years (aligned with NB7 / CLIMADA outputs)
# We only need the start year and the list of years for the CBA horizon.
ac_city_series = pd.read_csv(INT / f"ac_per_user_city_{SLUG}.csv")
ac_city_series = ac_city_series.set_index("year").sort_index()
ELEC_START_YEAR = int(ac_city_series.index.min()) # should be 2030
ELEC_YEARS = np.arange(ELEC_START_YEAR, ELEC_START_YEAR + T, dtype=int)

In [31]:
# AC COSTS: dynamic coverage + dynamic kWh/user

# AC PARAMS
AC_CAPEX_PER_USER = 500.0 # € per AC unit
AC_MAINT_RATE = 0.05 # fraction of CAPEX per year
AC_LIFETIME_YEARS = 10 # replacement cycle
TARIFF_EUR_PER_KWH = 0.25 # €/kWh

# per-user annual maintenance
maint_per_user_yr = AC_MAINT_RATE * AC_CAPEX_PER_USER

# Horizon years (should match benefits horizon: 2030..2030+T-1)
YEARS = ELEC_YEARS.copy()
assert len(YEARS) == T

# Municipio level coverage over time
# Table with coverage by year and municipality (cf notebook 5, we have it there)
try:
    muni_cov_all = pd.read_csv(OUT / f"{SLUG}_muni_cov_yearly.csv")
except FileNotFoundError:
    muni_cov_all = muni_cov.copy()

# we keep only rows for which coverage is defined
# and, we can, restrict to Municipi inside the city proper
if "muni_id" in muni_cov_all.columns:
    muni_cov_all = muni_cov_all.loc[muni_cov_all["muni_id"] > 0].copy()

# additional AC share per municipio and year (policy vs baseline)
muni_cov_all["dshare"] = (
    muni_cov_all["ac_policy_muni"] - muni_cov_all["ac_base_muni"]
).clip(lower=0.0)

# interpolating dshare to all years for each municipality
rows = []
for muni_id, g in muni_cov_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)
    known_dshare = g["dshare"].to_numpy(float)
    pop_muni = float(g["pop_muni"].iloc[0]) # assume pop_muni constant over time

    dshare_t = np.interp(YEARS, known_years, known_dshare) # flat before first and after last known year
    dshare_t[YEARS <= known_years[0]] = known_dshare[0]
    dshare_t[YEARS >= known_years[-1]] = known_dshare[-1]

    for year, ds in zip(YEARS, dshare_t):
        rows.append({
            "year": year,
            "muni_id": muni_id,
            "pop_muni": pop_muni,
            "dshare_t": ds,
        })

# coverage ramp
cov_yearly = pd.DataFrame(rows)

# total policy AC users per year (vs baseline)
added_users_t = (
    cov_yearly
    .assign(users=lambda d: d["pop_muni"] * d["dshare_t"])
    .groupby("year")["users"]
    .sum()
    .reindex(YEARS)
    .to_numpy(float)
)

# added_users_t: total incremental AC users each year vs baseline (from coverage ramp)
# new_users_t: year-to-year increments (cohorts) -> used for CAPEX with replacements
new_users_t = np.empty_like(added_users_t)
new_users_t[0] = added_users_t[0]
new_users_t[1:] = np.maximum(added_users_t[1:] - added_users_t[:-1], 0.0)
added_users_final = float(added_users_t[-1])
print(f"AC — added users in final year ≈ {added_users_final:,.0f}")

# Electricity costs: dynamic coverage + dynamic kWh/user from NB7

# muni-level kWh per AC user from NB7 (available for 2030, 2040, 2050)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")
years_full = YEARS # array([2030, ..., 2054])

rows_kwh = []
for muni_id, g in muni_tbl_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)
    vals = g["kwh_per_user_muni"].to_numpy(float)

    # interpolate to the full horizon
    kwh_interp = np.interp(years_full, known_years, vals) # flat before first and after last known year
    kwh_interp[years_full <= known_years[0]] = vals[0]
    kwh_interp[years_full >= known_years[-1]] = vals[-1]

    for y, v in zip(years_full, kwh_interp):
        rows_kwh.append({
            "year": y,
            "muni_id": muni_id,
            "kwh_per_user_muni": v,
        })

muni_kwh_full = pd.DataFrame(rows_kwh)

# attach interpolated kWh/user to coverage ramp
cov_yearly = cov_yearly.merge(
    muni_kwh_full, on=["year", "muni_id"], how="left"
).fillna({"kwh_per_user_muni": 0.0})

# kWh per year: pop * extra AC share * kWh per AC user
cov_yearly["kwh_t"] = (
    cov_yearly["pop_muni"] * cov_yearly["dshare_t"] * cov_yearly["kwh_per_user_muni"]
)

elec_eur_t = (
    cov_yearly.groupby("year")["kwh_t"].sum()
    .reindex(YEARS)
    .to_numpy(float)
) * TARIFF_EUR_PER_KWH

# discounted PV of elec, capex, maintenance
yrs = np.arange(1, T+1, dtype=float) # 1..25

# Electricity is a yearly flow: dynamic users * dynamic kWh/user * tariff
PV_ac_elec = float(np.sum(elec_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV elec (dynamic coverage & kWh/user): €{PV_ac_elec:,.0f}")

# maintenance on all active policy users
maint_eur_t = added_users_t * maint_per_user_yr
PV_ac_maint = float(np.sum(maint_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV maint €{PV_ac_maint:,.0f}")

# capex: cohorts of new users, with replacements every AC_LIFETIME_YEARS
PV_ac_capex = pv_capex_with_ramp(
    new_users_t,
    capex_per_user=AC_CAPEX_PER_USER,
    life=AC_LIFETIME_YEARS,
    r=R,
)
print(f"AC — PV capex €{PV_ac_capex:,.0f}")

# Total PV and EAC
PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec
print(f"AC — PV capex €{PV_ac_capex:,.0f}")
print(f"AC — PV maint €{PV_ac_maint:,.0f}")
print(f"AC — PV elec €{PV_ac_elec:,.0f}")
print(f"AC — PV total €{PV_ac_total:,.0f}")

EAC_ac_total = PV_ac_total / AF
print(f"AC — EAC total (annuity): €{EAC_ac_total:,.0f}/yr")

AC — added users in final year ≈ 199,063
AC — PV elec (dynamic coverage & kWh/user): €868,871,567
AC — PV maint €93,209,551
AC — PV capex €281,703,895
AC — PV capex €281,703,895
AC — PV maint €93,209,551
AC — PV elec €868,871,567
AC — PV total €1,243,785,014
AC — EAC total (annuity): €71,427,925/yr


In [32]:
print("First 5 years of elec_eur_t:", elec_eur_t[:5])
print("Last 5 years of elec_eur_t:", elec_eur_t[-5:])

First 5 years of elec_eur_t: [58227575.18681791 56972894.72012631 55718264.92995242 54463685.81629623
 53209157.37915776]
Last 5 years of elec_eur_t: [47960842.10582834 47960842.10582834 47960842.10582834 47960842.10582834
 47960842.10582834]


**Benefits and summary**

**Benefits: avoided heat-related deaths (CLIMADA outputs)** 

- Inputs:
    - CLIMADA provides avoided heat-related deaths for 2030, 2040, 2050 under:
      * Trees vs current AC baseline    (avo_trees)
      * AC policy vs baseline           (avo_ac)
      * Trees + AC policy vs current AC (avo_both)

- These are already *incremental* impacts relative to the current-AC baseline (and vegetation)

- Time profile:
    - We interpolate each of these three series to a yearly time profile over the
      25-year horizon (ex: 2030–2054).

- Trees vs current AC:
    - Trees do not deliver their full cooling effect immediately
    - We apply a smooth linear ramp over TREE_RAMP_YEARS (here 12 years) to the
      tree effect:
      * tree_ramp goes from 0 in year 1 to 1 by year 12
      * trees_yr_ramped = trees_full * tree_ramp

- AC vs baseline:
    - We assume the AC policy is *in place* from 2030 onwards (no ramp in benefits),
      so ac_yr is just the interpolated AC avoided-death series.

- Trees + AC:
    - both_full is the CLIMADA result for "trees + AC" vs the current AC baseline.
    - We decompose it as:
      * AC-only effect: ac_full
      * extra tree effect conditional on AC: trees_cond_yr = both_full - ac_full
    - The conditional tree effect is ramped in the same way:
      * both_yr_ramped = ac_yr + trees_cond_yr * tree_ramp

- Discounted vs cumulative benefits:
    - pv_of_stream(...) computes the *discounted* present value of each avoided-death
      stream (PV_b_tree, PV_b_ac, PV_b_both), which is useful for comparing timing of
      benefits.
      - For cost-per-avoided-death metrics, we use instead the *undiscounted* cumulative number of avoided deaths over the horizon:
        * CUM_b_tree, CUM_b_ac, CUM_b_both and then overwrite PV_b_* variables with these
          cumulative counts for compatibility with the rest of the code.

- Incremental effect of trees given AC:
  - PV_b_tree_cond and CUM_b_tree_cond capture the *extra* benefit of trees if the AC policy is already in place (both - AC).
  - This allows us to report a marginal cost-per-avoided-death for trees "on top" of an AC expansion.

In [33]:
# Benefits and summary
import numpy as np
import pandas as pd
from pathlib import Path

HORIZON_YEARS = T
DISCOUNT_RATE = R
TREE_RAMP_YEARS = 12

INT = Path(INT)
USE_SCALED_BENEFITS = False

def _nb6_path(stem: str) -> Path:
    return INT / (f"{stem}_scaled_{SLUG}.csv" if USE_SCALED_BENEFITS else f"{stem}_{SLUG}.csv")

def _read_overall(stem: str) -> pd.Series:
    p = _nb6_path(stem)
    if not p.exists():
        raise FileNotFoundError(f"Missing NB6 output: {p}")
    s = pd.read_csv(p, index_col="year")["overall"]
    s.index = s.index.astype(int)
    return s.sort_index()

def pv_of_stream(cashflows, r=DISCOUNT_RATE):
    yrs = np.arange(1, len(cashflows) + 1, dtype=float)
    return float(np.sum(np.asarray(cashflows, float) * (1 + r) ** (-yrs)))

def interpolate_to_horizon(s: pd.Series, years: np.ndarray) -> np.ndarray:
    s = s.sort_index()
    known = s.index.to_numpy(int)
    vals = s.to_numpy(float)
    out = np.interp(years, known, vals)
    out[years <= known[0]] = vals[0]
    out[years >= known[-1]] = vals[-1]
    return out

avo_trees = _read_overall("annual_heat_deaths_avoided_trees_curr_AC")
avo_ac = _read_overall("annual_heat_deaths_avoided_AC_curr_AC")
avo_both = _read_overall("annual_heat_deaths_avoided_treesplusAC_curr_AC")

try:
    avo_trees_on_top = _read_overall("annual_heat_deaths_avoided_trees_on_top_AC_curr_AC")
except FileNotFoundError:
    avo_trees_on_top = (avo_both - avo_ac).rename("overall")

# these are:
# If the full trees intervention (full ΔGVI map) were already in place
# (and effectively delivering its modeled cooling),
# what deaths would it avoid in each climate year?”
print("Trees – avoided deaths per year:")
print(avo_trees, "\n")
print("AC policy – avoided deaths per year:")
print(avo_ac, "\n")
print("Both (trees+AC vs current AC) – avoided deaths per year:")
print(avo_both, "\n")
print("Trees on top of AC (NB6 file) – avoided deaths per year:")
print(avo_trees_on_top)

# Horizon construction
START_YEAR = int(min(avo_trees.index.min(), avo_ac.index.min(), avo_both.index.min(), avo_trees_on_top.index.min()))
YEARS = np.arange(START_YEAR, START_YEAR + HORIZON_YEARS, dtype=int)

trees_full = interpolate_to_horizon(avo_trees, YEARS)
ac_full = interpolate_to_horizon(avo_ac, YEARS)
both_full = interpolate_to_horizon(avo_both, YEARS)
top_full = interpolate_to_horizon(avo_trees_on_top, YEARS)

# Trees benefits: cohort rollout (25y) + maturity (12y)
def cohort_rollout_maturity_factor(T, ramp_years, plant_share=None):
    if plant_share is None:
        plant_share = np.ones(T, dtype=float) / T # linear rollout over T years
    ages = np.arange(T, dtype=float) # 0..T-1 (cohort age in years)
    maturity = np.minimum((ages + 1) / ramp_years, 1.0)
    # factor[t] = sum_{i<=t} plant_share[i] * maturity[t-i]
    return np.convolve(plant_share, maturity)[:T]

# Benefit timing sensitivity: STATIC (full effect) vs DYNAMIC (rollout+maturity)
# Static ramp (linear 0→1) assumes trees give proportional benefits immediately
# as soon as “planted”. That’s too optimistic.
# Dynamic ramp matches what costs usually imply:
# cohorts planted each year + benefits grow as trees mature.
def compute_streams(trees_full, ac_full, both_full, top_full, trees_factor):
    """ Returns yearly benefit streams over the horizon.
    We keep the 'on-top' decomposition so that:
    both = ac + (trees-on-top)*factor
    """
    trees = trees_full * trees_factor
    ac = ac_full # no ramp factor applied
    top = top_full * trees_factor
    both = ac + top
    return trees, ac, top, both

trees_factor_dynamic = cohort_rollout_maturity_factor(T=HORIZON_YEARS, ramp_years=TREE_RAMP_YEARS)
trees_factor_static = np.ones(HORIZON_YEARS, dtype=float)

trees_dyn, ac_dyn, top_dyn, both_dyn = compute_streams(trees_full, ac_full, both_full, top_full, trees_factor_dynamic)
trees_sta, ac_sta, top_sta, both_sta = compute_streams(trees_full, ac_full, both_full, top_full, trees_factor_static)

def summarize_benefits(prefix, trees, ac, top, both, r=DISCOUNT_RATE):
    pv = {
        f"{prefix}_PV_trees": pv_of_stream(trees, r),
        f"{prefix}_PV_ac": pv_of_stream(ac, r),
        f"{prefix}_PV_top": pv_of_stream(top, r),
        f"{prefix}_PV_both": pv_of_stream(both, r),
    }
    cum = {
        f"{prefix}_CUM_trees": float(np.sum(trees)),
        f"{prefix}_CUM_ac": float(np.sum(ac)),
        f"{prefix}_CUM_top": float(np.sum(top)),
        f"{prefix}_CUM_both": float(np.sum(both)),
    }
    return pv, cum

pv_dyn, cum_dyn = summarize_benefits("DYN", trees_dyn, ac_dyn, top_dyn, both_dyn)
pv_sta, cum_sta = summarize_benefits("STA", trees_sta, ac_sta, top_sta, both_sta)

print("Cumulative avoided deaths (25y, undiscounted):")
print(
    f" DYNAMIC — Trees: {cum_dyn['DYN_CUM_trees']:.2f} | AC: {cum_dyn['DYN_CUM_ac']:.2f} | "
    f"Trees on top: {cum_dyn['DYN_CUM_top']:.2f} | Both: {cum_dyn['DYN_CUM_both']:.2f}"
)
print(
    f" STATIC — Trees: {cum_sta['STA_CUM_trees']:.2f} | AC: {cum_sta['STA_CUM_ac']:.2f} | "
    f"Trees on top: {cum_sta['STA_CUM_top']:.2f} | Both: {cum_sta['STA_CUM_both']:.2f}"
)
print("\nNote: 'STATIC' assumes full policy effect from the first year of the horizon.")
print(" 'DYNAMIC' assumes gradual rollout + tree maturity, so early benefits are smaller.")

Trees – avoided deaths per year:
year
2030    8.700125
2040    9.050604
2050    9.537256
Name: overall, dtype: float64 

AC policy – avoided deaths per year:
year
2030    32.185809
2040    28.150387
2050    30.050515
Name: overall, dtype: float64 

Both (trees+AC vs current AC) – avoided deaths per year:
year
2030    40.229810
2040    36.635876
2050    38.991879
Name: overall, dtype: float64 

Trees on top of AC (NB6 file) – avoided deaths per year:
year
2030    8.044001
2040    8.485489
2050    8.941364
Name: overall, dtype: float64
Cumulative avoided deaths (25y, undiscounted):
 DYNAMIC — Trees: 77.17 | AC: 744.01 | Trees on top: 72.33 | Both: 816.33
 STATIC — Trees: 228.96 | AC: 744.01 | Trees on top: 214.04 | Both: 958.05

Note: 'STATIC' assumes full policy effect from the first year of the horizon.
 'DYNAMIC' assumes gradual rollout + tree maturity, so early benefits are smaller.


- Baseline mortality and percentage reductions
- 
We now read the baseline heat-attributable deaths with current AC (no new policy) from CLIMADA and:
    - compute total baseline deaths per year (summing over age classes)
    - align avoided deaths for trees, AC, and both to these baseline years
    - compute percentage reductions in baseline deaths:
      * trees_pct = % reduction with trees only
      * ac_pct    = % reduction with AC only
      * both_pct  = % reduction with both policies
      * trees_on_top_pct = extra % reduction from adding trees on top of AC

In [34]:
# Baseline mortality
from pathlib import Path
import numpy as np
import pandas as pd

def read_baseline_curr_ac(int_dir: Path, slug: str) -> pd.DataFrame:
    """ Reads baseline annual heat-attributable deaths under current AC (no new policy)
    from the NB6 Excel bundle, sheet 'yr_totals_base'.

    Returns a DataFrame indexed by year with a single column: 'baseline_total'.
    """
    xlsx_path = Path(int_dir) / f"annual_heat_deaths_AC_bundle_{slug}.xlsx"
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Missing Excel bundle: {xlsx_path}")

    df = pd.read_excel(xlsx_path, sheet_name="yr_totals_base", index_col=0)
    df.index = df.index.astype(int)
    df = df.sort_index()

    if "overall" in df.columns:
        baseline = df["overall"].astype(float)
    else:
        # sum all numeric columns (typically age groups)
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) == 0:
            raise ValueError(f"'yr_totals_base' has no numeric columns to sum.")
        baseline = df[numeric_cols].sum(axis=1).astype(float)

    return baseline.to_frame("baseline_total")

baseline_df = read_baseline_curr_ac(INT, SLUG)

# align baseline to our horizon
baseline_total = interpolate_to_horizon(baseline_df["baseline_total"], YEARS)

def make_benefit_pct_table(label: str, baseline_total, trees, ac, both, top) -> pd.DataFrame:
    out = pd.DataFrame({
        "year": YEARS,
        "baseline_total": baseline_total,
        "avo_trees": trees,
        "avo_ac": ac,
        "avo_both": both,
        "avo_trees_on_top": top,
    }).set_index("year")
    out["trees_pct"] = 100 * out["avo_trees"] / out["baseline_total"]
    out["ac_pct"] = 100 * out["avo_ac"] / out["baseline_total"]
    out["both_pct"] = 100 * out["avo_both"] / out["baseline_total"]
    # two equivalent ways; this matches old logic
    out["trees_on_top_pct"] = out["both_pct"] - out["ac_pct"]
    # optional: tag timing convention
    out.insert(0, "Benefit_timing", label)
    return out

benefit_pct_dynamic = make_benefit_pct_table(
    "Dynamic rollout + tree maturity",
    baseline_total, trees_dyn, ac_dyn, both_dyn, top_dyn
).round(2)

benefit_pct_static = make_benefit_pct_table(
    "Static (full effect from start)",
    baseline_total, trees_sta, ac_sta, both_sta, top_sta
).round(2)

benefit_pct = pd.concat([benefit_pct_dynamic, benefit_pct_static])
benefit_pct

,Benefit_timing,baseline_total,avo_trees,avo_ac,avo_both,avo_trees_on_top,trees_pct,ac_pct,both_pct,trees_on_top_pct
year,,,,,,,,,,
2030,Dynamic rollout + tree maturity,643.65,0.03,32.19,32.21,0.03,0.00,5.00,5.00,0.00
2031,Dynamic rollout + tree maturity,648.58,0.09,31.78,31.86,0.08,0.01,4.90,4.91,0.01
2032,Dynamic rollout + tree maturity,653.51,0.18,31.38,31.54,0.16,0.03,4.80,4.83,0.02
2033,Dynamic rollout + tree maturity,658.44,0.29,30.98,31.25,0.27,0.04,4.70,4.75,0.04
2034,Dynamic rollout + tree maturity,663.37,0.44,30.57,30.98,0.41,0.07,4.61,4.67,0.06
2035,Dynamic rollout + tree maturity,668.30,0.62,30.17,30.75,0.58,0.09,4.51,4.60,0.09
2036,Dynamic rollout + tree maturity,673.24,0.83,29.76,30.54,0.78,0.12,4.42,4.54,0.12
2037,Dynamic rollout + tree maturity,678.17,1.07,29.36,30.36,1.00,0.16,4.33,4.48,0.15
2038,Dynamic rollout + tree maturity,683.10,1.35,28.96,30.22,1.26,0.20,4.24,4.42,0.18


- Summary table: costs, avoided deaths, and EACs by policy

- We assemble a compact table with one row per policy case:
1) "Trees only (vs current AC)"
2) "AC only (vs current AC)"
3) "Both (trees + AC vs current AC)"
4) "Trees (incremental, on top of AC policy)"

For each case we report:
- PV_cost_eur:
  * For trees: PV_trees_total (CAPEX + O&M)
  * For AC:    PV_ac_total (CAPEX + maintenance + electricity)
  * For "Both": sum of trees and AC PVs
- avoided_deaths_cum:
  * Cumulative (undiscounted) avoided deaths over the 25-year horizon.
- Cost_per_avoided_death_eur:
  * PV_cost_eur divided by avoided_deaths_cum.
  * This is a present-value cost divided by an *undiscounted* number of avoided deaths.
    We keep this mixed metric for interpretability (it is closer to a "cost per life
    saved over the period").
- EAC_*_annuity_eur_per_yr:
  * Equivalent annual costs derived from the PVs and the annuity factor AF.
  * For trees we separate CAPEX and O&M; for AC we use a single total EAC.
- EAC_capex_paper_eur_per_yr:
  * Trees-only: "paper-style" annual cost based on the shortcut IR / ((1+r)^T * T).
    This is kept only for comparability with the working paper, the main EAC
    we rely on is the annuity-based one using the actual ramp.
- added_AC_users:
  * Total number of new AC users in the last year of the horizon under the AC policy (for
    rows that include AC). This table is the main quantitative output used in the text to
    compare:
  - Trees vs AC in terms of cost per avoided death and annual cost
  - The combined policy vs each single policy
  -  The incremental value of trees when an AC expansion is already in place.

In [35]:
def safe_ratio(c, b):
    return float(c / b) if (b is not None and b > 1e-9) else np.inf

def build_summary(label, CUM_trees, CUM_ac, CUM_both, CUM_top):
    return pd.DataFrame([
        {
            "Benefit_timing": label,
            "Policy": "Trees only (vs current AC)",
            "PV_cost_eur": PV_trees_total,
            "avoided_deaths_cum": CUM_trees,
            "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total, CUM_trees),
            "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
            "EAC_om_annuity_eur_per_yr": EAC_om_annuity,
            "EAC_total_annuity_eur_per_yr": EAC_total_annuity,
            "EAC_capex_paper_eur_per_yr": EAC_capex_paper,
            "added_AC_users": 0.0,
        },
        {
            "Benefit_timing": label,
            "Policy": "AC only (vs current AC)",
            "PV_cost_eur": PV_ac_total,
            "avoided_deaths_cum": CUM_ac,
            "Cost_per_avoided_death_eur": safe_ratio(PV_ac_total, CUM_ac),
            "EAC_capex_annuity_eur_per_yr": 0.0,
            "EAC_om_annuity_eur_per_yr": 0.0,
            "EAC_total_annuity_eur_per_yr": EAC_ac_total,
            "EAC_capex_paper_eur_per_yr": np.nan,
            "added_AC_users": added_users_final,
        },
        {
            "Benefit_timing": label,
            "Policy": "Both (trees+AC vs current AC)",
            "PV_cost_eur": PV_trees_total + PV_ac_total,
            "avoided_deaths_cum": CUM_both,
            "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total + PV_ac_total, CUM_both),
            "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
            "EAC_om_annuity_eur_per_yr": EAC_om_annuity,
            "EAC_total_annuity_eur_per_yr": EAC_total_annuity + EAC_ac_total,
            "EAC_capex_paper_eur_per_yr": EAC_capex_paper,
            "added_AC_users": added_users_final,
        },
        {
            "Benefit_timing": label,
            "Policy": "Trees (incremental, on top of AC policy)",
            "PV_cost_eur": PV_trees_total,
            "avoided_deaths_cum": CUM_top,
            "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total, CUM_top),
            "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
            "EAC_om_annuity_eur_per_yr": EAC_om_annuity,
            "EAC_total_annuity_eur_per_yr": EAC_total_annuity,
            "EAC_capex_paper_eur_per_yr": EAC_capex_paper,
            "added_AC_users": 0.0,
        },
    ])

summary_dynamic = build_summary(
    "Dynamic rollout + tree maturity",
    cum_dyn["DYN_CUM_trees"], cum_dyn["DYN_CUM_ac"], cum_dyn["DYN_CUM_both"], cum_dyn["DYN_CUM_top"]
).round(2)

summary_static = build_summary(
    "Static (full effect from start)",
    cum_sta["STA_CUM_trees"], cum_sta["STA_CUM_ac"], cum_sta["STA_CUM_both"], cum_sta["STA_CUM_top"]
).round(2)

summary_both = pd.concat([summary_dynamic, summary_static], ignore_index=True)
summary_both

,Benefit_timing,Policy,PV_cost_eur,avoided_deaths_cum,Cost_per_avoided_death_eur,EAC_capex_annuity_eur_per_yr,EAC_om_annuity_eur_per_yr,EAC_total_annuity_eur_per_yr,EAC_capex_paper_eur_per_yr,added_AC_users
0,Dynamic rollout + tree maturity,Trees only (vs current AC),4.942419e+08,77.17,6404340.47,12093350.58,16289910.33,28383260.92,5775851.59,0.00
1,Dynamic rollout + tree maturity,AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,71427925.36,NaN,199063.06
2,Dynamic rollout + tree maturity,Both (trees+AC vs current AC),1.738027e+09,816.33,2129071.53,12093350.58,16289910.33,99811186.28,5775851.59,199063.06
3,Dynamic rollout + tree maturity,"Trees (incremental, on top of AC policy)",4.942419e+08,72.33,6833604.62,12093350.58,16289910.33,28383260.92,5775851.59,0.00
4,Static (full effect from start),Trees only (vs current AC),4.942419e+08,228.96,2158632.48,12093350.58,16289910.33,28383260.92,5775851.59,0.00
5,Static (full effect from start),AC only (vs current AC),1.243785e+09,744.01,1671741.22,0.00,0.00,71427925.36,NaN,199063.06
6,Static (full effect from start),Both (trees+AC vs current AC),1.738027e+09,958.05,1814138.07,12093350.58,16289910.33,99811186.28,5775851.59,199063.06
7,Static (full effect from start),"Trees (incremental, on top of AC policy)",4.942419e+08,214.04,2309111.62,12093350.58,16289910.33,28383260.92,5775851.59,0.00


**On a PV budget**

**Sensitivity**